In [23]:
from google.colab import files

uploaded = files.upload()

Saving malayalam_glossary.pdf to malayalam_glossary.pdf


In [6]:
!pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 28.3 MB/s eta 0:00:00


In [6]:
!pip install -q pandas==2.2.3

In [8]:
!pip install -q -U transformers accelerate sentencepiece pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 116.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


In [1]:
from huggingface_hub import login

login()

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "google/gemma-3-1b-it"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading Gemma 3 1B...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Gemma 3 1B loaded successfully!")
print("Device:", model.device)

Loading tokenizer...
Loading Gemma 3 1B...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Gemma 3 1B loaded successfully!
Device: cuda:0


In [25]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [26]:
import fitz

pdf_file = "malayalam_glossary.pdf"

doc = fitz.open(pdf_file)

text = ""

for page in doc:
    text += page.get_text()

print(text)

Basic Computer & Technology Glossary
Context: Computer Science and Information Technology
No.
English Term
1
Algorithm
2
Internet
3
Data
4
Memory
5
Network
6
Website
7
Email
8
Technology
9
Web
10
Printer
This English glossary is used as input to a Small Language Model (Gemma 3 1B Instruct) for
English-to-Malayalam translation.



In [27]:
import re

# Use the text extracted from the PDF
# Your previous cell appears to have stored it in 'text'
pdf_text = text

# Extract terms from the table format:
# number appears on one line and the term on the next line
lines = [line.strip() for line in pdf_text.splitlines() if line.strip()]

terms = []

for i in range(len(lines) - 1):
    if re.fullmatch(r'\d+', lines[i]):
        number = int(lines[i])

        # Only accept glossary numbers 1–20
        if 1 <= number <= 20:
            term = lines[i + 1]

            # Avoid accidentally including the note at the end
            if not term.startswith("Note:"):
                terms.append(term)

# Remove duplicates while preserving order
terms = list(dict.fromkeys(terms))

print("Number of terms:", len(terms))
print("\nTerms extracted from PDF:\n")

for i, term in enumerate(terms, 1):
    print(f"{i}. {term}")

Number of terms: 10

Terms extracted from PDF:

1. Algorithm
2. Internet
3. Data
4. Memory
5. Network
6. Website
7. Email
8. Technology
9. Web
10. Printer


In [29]:
def translate_to_malayalam(term):

    messages = [
        {
            "role": "system",
            "content": """You are an English to Malayalam translation system.

Your ONLY task is to translate Computer Science terms into Malayalam.

Malayalam uses this script:
അ ആ ഇ ഈ ഉ ഊ എ ഏ ഐ ഒ ഓ ക ഖ ഗ ഘ ങ ച ജ ട ഡ ത ദ ന പ ബ മ യ ര ല വ ശ സ ഹ

You MUST output Malayalam script.
Never output Chinese, Japanese, Hindi, Tamil, Kannada, Russian,
Arabic, or English.

Use commonly understood Malayalam transliterations for technical
Computer Science terms.

Examples:
Algorithm = അൽഗോരിതം
Database = ഡാറ്റാബേസ്
Computer = കമ്പ്യൂട്ടർ
Software = സോഫ്റ്റ്‌വെയർ
Hardware = ഹാർഡ്‌വെയർ
Internet = ഇന്റർനെറ്റ്
Programming = പ്രോഗ്രാമിംഗ്
Network = നെറ്റ്‌വർക്ക്
Technology = സാങ്കേതികവിദ്യ
Data = ഡാറ്റ

Return ONLY the Malayalam translation."""
        },
        {
            "role": "user",
            "content": f"Translate this Computer Science term:\n{term}"
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    result = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return result

In [32]:
final_terms = [
    "Algorithm",
    "Internet",
    "Data",
    "Memory",
    "Network",
    "Website",
    "Email",
    "Technology",
    "Web"
]

print("=" * 70)
print("FINAL MALAYALAM TRANSLATION")
print("=" * 70)

for term in final_terms:
    translation = translate_to_malayalam(term)
    print(f"{term:15} → {translation}")

FINAL MALAYALAM TRANSLATION
Algorithm       → അൽഗോരിതം
Internet        → ഇന്റർനെറ്റ്
Data            → ഡാറ്റ
Memory          → മെമറി
Network         → നെറ്റ്‌വർക്ക്
Website         → വെബ്സൈറ്റ്
Email           → ഇമെയിൽ
Technology      → സാങ്കേതികവിദ്യ
Web             → വെബ്


In [2]:
import pandas as pd

results = [
    ["Algorithm", "അൽഗോരിതം"],
    ["Internet", "ഇന്റർനെറ്റ്"],
    ["Data", "ഡാറ്റ"],
    ["Memory", "മെമറി"],
    ["Network", "നെറ്റ്‌വർക്ക്"],
    ["Website", "വെബ്സൈറ്റ്"],
    ["Email", "ഇമെയിൽ"],
    ["Technology", "സാങ്കേതികവിദ്യ"],
    ["Web", "വെബ്"]
]

df = pd.DataFrame(results, columns=["English", "Malayalam"])

df.to_csv("english_malayalam_glossary.csv", index=False, encoding="utf-8-sig")

print(df.to_string(index=False))

   English      Malayalam
 Algorithm       അൽഗോരിതം
  Internet    ഇന്റർനെറ്റ്
      Data          ഡാറ്റ
    Memory          മെമറി
   Network  നെറ്റ്‌വർക്ക്
   Website     വെബ്സൈറ്റ്
     Email         ഇമെയിൽ
Technology സാങ്കേതികവിദ്യ
       Web           വെബ്


In [3]:
df.to_csv(
    "english_malayalam_glossary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV file created successfully!")

CSV file created successfully!
